In [0]:
RUNDIR = 'u/dobos/dobos-test-123411/20260728T152502Z'
RUN = RUNDIR.replace('/', '_')
DATE = '20250402'
VISIT = '123411'
ARM = 'm'
SPECTROGRAPH = '1'
FIBERID = 130

PFSARM_PATH = f'/datascope/subaru/data/datastore/{RUNDIR}/pfsArm/{DATE}/{VISIT}/pfsArm_PFS_{VISIT}_{ARM}{SPECTROGRAPH}_{RUN}.fits'
LINES_PATH = f'/datascope/subaru/data/datastore/{RUNDIR}/lines/{DATE}/{VISIT}/lines_PFS_{VISIT}_{ARM}{SPECTROGRAPH}_{RUN}.fits'

In [0]:
import os
import re
from glob import glob

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from astropy.io import fits

In [0]:
with fits.open(PFSARM_PATH) as hdul:
    hdul.info()
    arm_fiberid = hdul['FIBERID'].data
    arm_wave = hdul['WAVELENGTH'].data

arm_fiberid.shape, arm_wave.shape

In [0]:
with fits.open(LINES_PATH) as hdul:
    hdul.info()
    lines = hdul['ARCLINES'].data

lines.columns

In [0]:
arm_fiberid

In [0]:
np.unique(lines.source)

# Plot the wavelength solution and the lines

In [0]:
ip(0), ip(arm_wave.shape[1])

In [0]:
fid = FIBERID

fig, ax = plt.subplots(figsize=(8, 5), dpi=240)

# Wavelength solution from pfsArm
i = np.where(arm_fiberid == fid)[0][0]
y = np.arange(arm_wave.shape[1])
w = arm_wave[i, :]
ip = interp1d(y, w, kind='linear', bounds_error=False, fill_value='extrapolate')

# ax.plot(y, w)
ax.axhline(0, color='k', lw=0.5, alpha=0.5)

# Line data from lines
# m = (lines.fiberId == fid) & (lines.source == 32)
m = (lines.fiberId == fid) & (lines.source == 0)

y = lines.y[m]
y_error = lines.yErr[m]
w = lines.wavelength[m] - ip(y)
w_error = ip(y + y_error) - ip(y - y_error)
c = np.log(lines.flux[m] + 1e-3)

# ax.plot(y, w, '+', markersize=3)
# ax.errorbar(y, w, yerr=w_error, fmt='.', markersize=0, elinewidth=0.5)
l = ax.scatter(y, w, c=c, s=4, cmap='jet', alpha=1)

ax.set_xlim(0, 4200)
# ax.set_ylim(-0.35, 0.35)
ax.set_ylim(-0.001, 0.001)
ax.set_xlabel('$y$ (pixel)')
ax.set_ylabel(R'$\Delta\lambda$ (nm)')

# Add a secondary x-axis for wavelength based on the wavelength solution from pfsArm
# ip(y) converts pixel to wavelength
ax2 = ax.twiny()
ax2.set_xlim(ip(0), ip(4200))
ax2.set_xlabel(R'$\lambda$ (nm)')

ax.set_title(f'RUN={RUN}\nVISIT={VISIT}, ARM={ARM}, SPECTROGRAPH={SPECTROGRAPH}, FIBERID={FIBERID}')
fig.colorbar(l, ax=ax, label='log flux')

In [0]:
lines.columns

In [0]:
m = (lines.fiberId == fid) & (lines.source == 0)
lines.flag[m]